In [2]:
path_imagem = 'pessoas-tirando-uma-selfie-juntas-no-dia-do-registro_23-2149096795.avif'

img = cv2.imread(path_imagem)

In [12]:
from face_detection import *
from face_utils import *
import mediapipe as mp

import cv2
cap = cv2.VideoCapture(0)

resize_factor = 0.333



_, frame = cap.read()
cv2.imshow('frame', frame)

cap.release()


cv2.waitKey(0)          # aguarda pressionar qualquer tecla
cv2.destroyAllWindows()


In [1]:
import mediapipe as mp
import numpy as np

mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=False,
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

def get_landmarks(frame):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb)
    
    if not results.multi_face_landmarks:
        return None
    
    h, w = frame.shape[:2]
    landmarks = np.array([
        [lm.x * w, lm.y * h]
        for lm in results.multi_face_landmarks[0].landmark
    ])
    
    return landmarks

get_landmarks(frame)

AttributeError: module 'mediapipe' has no attribute 'solutions'

In [2]:
import cv2
import mediapipe as mp

from mediapipe.tasks.python import vision
from mediapipe.tasks import python

# modelo
model_path = "face_landmarker.task"

base_options = python.BaseOptions(model_asset_path=model_path)

options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    output_face_blendshapes=True,
    output_facial_transformation_matrixes=False,
    num_faces=1
)

detector = vision.FaceLandmarker.create_from_options(options)

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    result = detector.detect(mp_image)

    if result.face_blendshapes:
        # índices dos olhos fechando (EAR-like via blendshapes)
        left_eye = result.face_blendshapes[0][9].score
        right_eye = result.face_blendshapes[0][10].score

        # quando piscando, valores sobem
        if left_eye > 0.5 and right_eye > 0.5:
            cv2.putText(frame, "PISCANDO", (30, 50),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    cv2.imshow("Blink Detection", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

In [1]:
from face_detection import *
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    eye_state = get_eye_state(frame)

    
    cv2.putText(frame, eye_state, (30, 50),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    cv2.imshow("Blink Detection", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

In [1]:
from pathlib import Path
from face_recogtion import *

registry = get_all_records()
registry

[Record id: 0 - ]

In [4]:
path_faces_test = Path('faces_pictures')

for face_picutre in path_faces_test.iterdir():
    name = face_picutre.stem
    if name == 'enola':
        continue
    register_face(face_picutre, name )

In [10]:
face_picutre.stem

'terry_cris'

In [4]:
clear_registry()

In [2]:
rec_eleven = get_records_by_name('eleven')[0]
emb_eleven = rec_eleven.embedding
rec_jack = get_records_by_name('jack')[0]
emb_jack = rec_jack.embedding

In [4]:
emb_enola = embed_face_image('faces_pictures/enola.jpeg')

In [17]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
emb_enola
emb_eleven
# similarity = cosine_similarity(emb_enola, emb_eleven)
def similarity(emb_1, emb_2):
    aemb_1 = np.array(emb_1)
    aemb_2 = np.array(emb_2)
    similarity = np.dot(aemb_1, aemb_2) / (
        np.linalg.norm(aemb_1) * np.linalg.norm(aemb_2)
    )
    return similarity
similarity(emb_enola,emb_jack)

np.float64(-0.02530127378581648)

In [6]:
get_record_by_emb(emb_jack)

Record id: 3 - jackchan

In [12]:
similarity(emb_enola, emb_eleven)

np.float64(0.6052605618708454)